In [ ]:
# Google Colab setup (uncomment if needed)
# from google.colab import drive
# drive.mount("/gdrive")
# %cd "/gdrive/My Drive/Challenge2"

## ⚙️ Setup

In [ ]:
# Install dependencies if needed
# !pip install pyyaml tensorboard scikit-learn pandas matplotlib seaborn tqdm

In [ ]:
import sys
import os
from pathlib import Path

# Add utils to path
sys.path.insert(0, str(Path.cwd()))

# Import everything from our streamlined package
# from utils import (
#     Config, set_seed, setup_device, create_dirs,
#     get_transforms, create_dataloaders,
#     get_model, Trainer,
#     evaluate_model, create_submission,
#     plot_training_history, plot_confusion_matrix
# )

import pandas as pd
import torch
from sklearn.model_selection import train_test_split

print("✅ All imports successful")

ModuleNotFoundError: No module named 'cv2'

## 📝 Configuration

**SINGLE SOURCE OF TRUTH** - Edit `config.yaml` or modify Config here

In [ ]:
# Option 1: Load from YAML file (recommended)
try:
    config = Config.from_yaml('config.yaml')
    print("✅ Loaded config from config.yaml")
except:
    print("⚠️  config.yaml not found, using defaults")
    config = Config()

# Option 2: Override specific settings
# config.NUM_EPOCHS = 30
# config.BATCH_SIZE = 32
# config.LEARNING_RATE = 0.001
# config.MODEL_NAME = 'resnet50'
# config.USE_TENSORBOARD = True

# Save current config for reproducibility
config.save_yaml('current_run_config.yaml')

print(f"\n📋 Config:")
print(f"   Model: {config.MODEL_NAME}")
print(f"   Epochs: {config.NUM_EPOCHS}")
print(f"   Batch: {config.BATCH_SIZE}")
print(f"   LR: {config.LEARNING_RATE}")

## 🎲 Reproducibility

In [ ]:
set_seed(config.SEED)
device = setup_device()
create_dirs(config)

print(f"✅ Seed: {config.SEED} | Device: {device}")

## 📊 Data Preparation

In [ ]:
# Load CSV
df = pd.read_csv(config.CSV_PATH)
print(f"✅ Loaded {len(df)} samples from {config.CSV_PATH}")

# Split train/val
train_df, val_df = train_test_split(
    df, 
    test_size=config.VAL_SPLIT,
    stratify=df['label'],
    random_state=config.SEED
)

print(f"   Train: {len(train_df)} | Val: {len(val_df)}")
print(f"\nClass distribution:")
print(train_df['label'].value_counts().sort_index())

In [ ]:
# Create data loaders
train_loader, val_loader, num_classes = create_dataloaders(
    train_df=train_df,
    val_df=val_df,
    data_dir=config.DATA_DIR,
    config=config,
    use_imagenet_norm=config.USE_IMAGENET_NORM
)

print(f"✅ Dataloaders created")
print(f"   Training batches: {len(train_loader)}")
print(f"   Validation batches: {len(val_loader)}")
print(f"   Number of classes detected: {num_classes}")

## 🏗️ Model

In [ ]:
# Build model
config.NUM_CLASSES = num_classes  # Use detected number of classes
model = get_model(config)
model = model.to(device)

# Model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Model: {config.MODEL_NAME}")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

## 🚀 Training

TensorBoard will log automatically if `USE_TENSORBOARD=True`

In [ ]:
# Start TensorBoard in background
if config.USE_TENSORBOARD:
    %load_ext tensorboard
    %tensorboard --logdir {config.TENSORBOARD_DIR}

In [ ]:
# Create trainer - all settings from config
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    device=device
)

# Train (supports auto-resume)
history = trainer.train(resume=False)

print("\n✅ Training complete!")

## 📈 Results

In [ ]:
# Plot training curves
history_df = trainer.get_history_df()
display(history_df.tail())

plot_training_history(history_df)

In [ ]:
# Evaluate on validation set
results = evaluate_model(
    model, 
    val_loader, 
    device,
    class_names=[f'Class_{i}' for i in range(num_classes)]
)

print(f"\n📊 Validation Results:")
print(f"   Accuracy: {results['accuracy']:.4f}")
print(f"   F1 Score: {results['f1']:.4f}")

# Confusion matrix
plot_confusion_matrix(
    results['confusion_matrix'],
    class_names=[f'Class_{i}' for i in range(num_classes)]
)

## 💾 Create Submission

In [ ]:
# Load best model
exp_name = config.EXPERIMENT_NAME
best_model_path = f"{config.MODELS_DIR}/{exp_name}_best.pt"
model.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=True))
print(f"✅ Loaded best model from {best_model_path}")

# Create submission (with masks if available)
submission_df = create_submission(
    model=model,
    test_dir=config.TEST_DIR,
    config=config,
    device=device,
    output_path='submission.csv',
    use_mask=True  # Apply masks to test images if available
)

print(f"✅ Submission saved to submission.csv")

---

## 🔧 Advanced: Two-Stage Training

Optionally train in two stages: frozen backbone first, then fine-tune

In [ ]:
# Stage 1: Train classifier only (frozen backbone)
config.FREEZE_BACKBONE = True
config.NUM_EPOCHS = 10
config.LEARNING_RATE = 0.001

model_stage1 = get_model(config)
trainer_stage1 = Trainer(model_stage1, train_loader, val_loader, config, device)
history_stage1 = trainer_stage1.train()

print("✅ Stage 1 complete (frozen backbone)")

In [ ]:
# Stage 2: Fine-tune entire model
from utils import freeze_backbone

freeze_backbone(model_stage1, freeze=False)  # Unfreeze
config.NUM_EPOCHS = 20
config.LEARNING_RATE = 0.0001  # Lower LR for fine-tuning

trainer_stage2 = Trainer(model_stage1, train_loader, val_loader, config, device)
history_stage2 = trainer_stage2.train()

print("✅ Stage 2 complete (fine-tuned)")